# Training Ulang Model dari Hyperparameter Tersimpan

Notebook ini melanjutkan `code_run_all.ipynb` **tanpa menjalankan Genetic Algorithm lagi**. Hyperparameter LightGBM dan XGBoost dibaca dari file `.pkl` hasil GA, lalu dipakai untuk melatih ulang, mengevaluasi, dan menyimpan model.

Alur: muat data → imputasi → `log1p` → *differencing* orde 1 → `ForecasterDirect` dengan parameter dari `.pkl` → evaluasi test → backtesting walk-forward → Early Warning System H+30 → simpan final model.

Pipeline-nya sendiri berada di `research_common.py` dan dipakai bersama `code_run_all.ipynb`, sehingga split train/test dan konfigurasi fitur dijamin identik dengan saat GA dijalankan.

Prasyarat: `models/genetic_algorithm_parameters.pkl` (LightGBM) dan/atau `models/genetic_algorithm_parameters_xgboost.pkl` (XGBoost) sudah ada dari `code_run_all.ipynb`.

## 1. Import Library

Komponen pipeline diambil dari `research_common.py`; notebook ini hanya perlu `joblib` untuk membaca `.pkl` dan `pandas` untuk merangkum hasil.

In [ ]:
import warnings
from pathlib import Path

import joblib as jl
import pandas as pd
from skforecast.exceptions import IgnoredArgumentWarning, LongTrainingWarning

import research_common as rc

In [ ]:
warnings.simplefilter("ignore", category=IgnoredArgumentWarning)
warnings.simplefilter("ignore", category=LongTrainingWarning)

## 2. Memuat Dataset, Split Train/Test, dan Transformasi Log

`rc.ResearchData.from_csv` menjalankan langkah yang identik dengan `code_run_all.ipynb`: indeks harian, fold terakhir `TimeSeriesSplit(n_splits=5)`, train diimputasi interpolasi waktu dan test *forward fill*, lalu `log1p(harga)` disiapkan sebagai target model. Prediksi dikembalikan ke skala harga dengan `expm1`.

In [ ]:
R = rc.ResearchData.from_csv("DATASET-BERAS.csv")
print(f"{len(R.commodity_cols)} komoditas siap dilatih")

## 3. Konfigurasi Fitur dan Forecaster

Lag 1/7/30 hari, rolling mean-std-min-max jendela 7 dan 30 hari, fitur kalender siklikal, dan *differencing* orde 1 di dalam `ForecasterDirect` — persis seperti konfigurasi saat GA mencari parameter, karena keduanya memakai `research_common.py`.

In [ ]:
print("Horizon         :", R.horizons)
print("Lag             :", R.lags)
print("Differencing    :", R.differentiation)
print("Rolling stats   :", R.window_features.stats)
print("Rolling windows :", R.window_features.window_sizes)
print("Kalender        :", R.calendar_features.features)

## 4. Memuat Hyperparameter Hasil GA (Tanpa Pencarian Ulang)

Parameter dibaca dari `.pkl` hasil `code_run_all.ipynb`. Struktur file: `params[komoditas][horizon]["params"]`. Model yang file parameternya belum ada otomatis dilewati, jadi notebook tetap jalan meski baru satu model yang dioptimasi.

In [ ]:
MODEL_SPECS = {
    "LightGBM": {
        "params_path": "models/genetic_algorithm_parameters.pkl",
        "estimator": rc.make_lgb,
        "forecaster_dir": "final_model",
        "prefix": "final_model_prediction",
        "color": "#d97941",
    },
    "XGBoost": {
        "params_path": "models/genetic_algorithm_parameters_xgboost.pkl",
        "estimator": rc.make_xgb,
        "forecaster_dir": "final_model_xgb",
        "prefix": "final_model_prediction_xgb",
        "color": "#2b5c8f",
    },
}

best_params = {}
for model_name, spec in MODEL_SPECS.items():
    params_path = Path(spec["params_path"])
    if params_path.exists():
        best_params[model_name] = jl.load(params_path)
        print(f"{model_name}: parameter dimuat dari {params_path}")
    else:
        print(
            f"{model_name}: {params_path} belum ada — dilewati "
            f"(jalankan GA di code_run_all.ipynb)"
        )

assert best_params, (
    "Tidak ada file parameter GA. Jalankan pencarian di code_run_all.ipynb dulu."
)

missing = [
    (model_name, col, h)
    for model_name, params_store in best_params.items()
    for col in R.commodity_cols
    for h in R.horizons
    if h not in params_store.get(col, {})
]
assert not missing, f"Parameter tidak lengkap untuk: {missing[:5]}"
print(f"\nModel aktif: {', '.join(best_params)}")

## 5. Menampilkan Hyperparameter yang Dipakai

Konfirmasi parameter yang dibaca dari `.pkl` sebelum melatih.

In [ ]:
for model_name, params_store in best_params.items():
    print("\n" + "#" * 72)
    print(f"MODEL: {model_name}")
    print("#" * 72)
    rc.print_params_block(params_store, R.horizons)

## 6. Melatih Ulang Forecaster dari Parameter Tersimpan

Setiap kombinasi model × komoditas × horizon dilatih pada log harga data train.

In [ ]:
forecasters = {}

for model_name, params_store in best_params.items():
    forecasters[model_name] = R.fit_from_params(
        params_store, MODEL_SPECS[model_name]["estimator"]
    )
    print(
        f"{model_name}: {len(R.commodity_cols) * len(R.horizons)} forecaster terlatih "
        f"({len(R.commodity_cols)} komoditas × {len(R.horizons)} horizon)"
    )

## 7. Evaluasi pada Data Test

Prediksi `h` hari pertama data test, dikembalikan ke skala harga dengan `expm1` sebelum MAE, RMSE, dan MAPE dihitung.

In [ ]:
df_eval = pd.concat(
    [
        R.test_metrics(model_forecasters, extra={"Model": model_name})
        for model_name, model_forecasters in forecasters.items()
    ],
    ignore_index=True,
)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
df_eval

## 8. Ringkasan MAPE Antar Model

Pivot MAPE per komoditas dan horizon untuk membandingkan model secara langsung.

In [ ]:
df_mape_pivot = df_eval.pivot_table(
    index=["Komoditas", "Horizon"], columns="Model", values="MAPE (%)"
)
if len(df_mape_pivot.columns) > 1:
    df_mape_pivot["Model Terbaik"] = df_mape_pivot.idxmin(axis=1)
df_mape_pivot

## 9. Backtesting Walk-forward

Model dilatih sekali lalu memprediksi blok `h` hari berturut-turut sepanjang periode test dengan harga aktual terbaru sebagai input. Galat hanya dihitung pada tanggal yang punya observasi asli (bukan hasil imputasi).

In [ ]:
walkforward_metrics = {}
walkforward_predictions = {}

for model_name, params_store in best_params.items():
    metrics, predictions = R.walk_forward_evaluation(
        params_store, MODEL_SPECS[model_name]["estimator"]
    )
    walkforward_metrics[model_name] = metrics.assign(Model=model_name)
    walkforward_predictions[model_name] = predictions
    print(f"{model_name}: walk-forward selesai")

df_walkforward = pd.concat(walkforward_metrics.values(), ignore_index=True)
df_walkforward

## 10. Plot Harga Aktual vs Prediksi Walk-forward

Setiap panel membandingkan harga aktual dengan prediksi tiap model untuk satu komoditas dan horizon.

In [ ]:
R.plot_walkforward(
    {
        f"Prediksi {model_name}": (
            predictions,
            MODEL_SPECS[model_name]["color"],
        )
        for model_name, predictions in walkforward_predictions.items()
    },
    "Harga Aktual vs Prediksi Walk-forward",
)

## 11. Early Warning System H+30

Bandingkan prediksi H+30 dengan harga terakhir pada data train. Status: Waspada (> 5%), Warning (> 10%), dan Bahaya (> 20%).

In [ ]:
h_ews = 30

df_ews = pd.concat(
    [
        R.ews_table(model_forecasters, h_ews=h_ews, model_name=model_name)
        for model_name, model_forecasters in forecasters.items()
    ],
    ignore_index=True,
)
df_ews

## 12. Menyimpan Final Model

Simpan forecaster hasil training ulang: LightGBM ke `final_model`, XGBoost ke `final_model_xgb`.

In [ ]:
for model_name, model_forecasters in forecasters.items():
    spec = MODEL_SPECS[model_name]
    print(f"\n== {model_name} -> {spec['forecaster_dir']} ==")
    R.save_forecasters(model_forecasters, spec["forecaster_dir"], spec["prefix"])